# 🌍 Qdrant Multilingual Search Demo

## What You'll Learn

This notebook demonstrates **cross-lingual search** capabilities - the ability to:
- Query in one language and find results in other languages
- Store documents in multiple languages in the same vector store
- Understand how multilingual embeddings work

## Why This Matters

In a globalized world, your data might be in multiple languages:
- Customer support tickets in English, Spanish, French
- Product documentation translated to many languages
- International news articles

**Without multilingual search:** You'd need separate searches for each language.

**With multilingual search:** One query finds relevant content regardless of language!

## How It Works

```
English query: "artificial intelligence"
         ↓
    [Embedding Model]
         ↓
    Vector: [0.23, -0.45, 0.12, ...]
         ↓
    Matches vectors for:
    • "artificial intelligence" (English)
    • "inteligencia artificial" (Spanish)  
    • "intelligenza artificiale" (Italian)
```

The magic is in the **multilingual embedding model** which maps similar concepts to nearby vectors regardless of language.

## Prerequisites

1. ✅ Ollama running with `nomic-embed-text:latest` model
2. ✅ Qdrant running on port 6333
3. ✅ Llama Stack running on port 8321 with Qdrant configured:
   ```bash
   OLLAMA_URL=http://localhost:11434/v1 QDRANT_URL=http://localhost:6333 llama stack run starter --port 8321
   ```

---
## Step 1: Connect to Llama Stack

**What we're doing:** Establishing our connection to the Llama Stack server.

**Expected output:** Success message confirming connection.

In [ ]:
import io
from pathlib import Path
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8321/v1/",
    api_key="none"
)

print("✅ Connected to Llama Stack!")

---
## Step 2: Load Multilingual Documents

**What we're doing:** Loading sample articles about "AI in Healthcare" and "Climate Change" in three languages:
- 🇬🇧 English
- 🇪🇸 Spanish  
- 🇮🇹 Italian

**Why these documents:** They cover the same topics in different languages, making it easy to verify that cross-lingual search works correctly.

**Expected output:** The content of each document displayed.

In [ ]:
# Define the languages and topics we'll use
languages = ["en", "es", "it"]
topics = ["tech_article", "climate_article"]

language_names = {"en": "🇬🇧 English", "es": "🇪🇸 Spanish", "it": "🇮🇹 Italian"}

# Load all documents
documents = {}
base_path = Path("../data/multilingual")

print("📄 Loading multilingual documents...\n")

for lang in languages:
    for topic in topics:
        file_path = base_path / lang / f"{topic}.txt"
        if file_path.exists():
            with open(file_path) as f:
                content = f.read()
                key = f"{lang}_{topic}"
                documents[key] = {
                    "content": content,
                    "language": lang,
                    "topic": topic
                }

print(f"✅ Loaded {len(documents)} documents\n")

# Show a sample from each language
for lang in languages:
    key = f"{lang}_tech_article"
    if key in documents:
        preview = documents[key]['content'][:200]
        print(f"{language_names[lang]}:")
        print(f"   {preview}...\n")

---
## Step 3: Create Vector Store with Multilingual Embedding Model

**What we're doing:** Creating a vector store configured with an embedding model that has multilingual capabilities.

**Why `nomic-embed-text`:** While not a dedicated multilingual model, it has been trained on diverse data and provides reasonable cross-lingual understanding. For production multilingual applications, consider dedicated multilingual models.

**Expected output:** Vector store ID for subsequent operations.

In [ ]:
# Create vector store with embedding model
# Note: nomic-embed-text has some multilingual capability
vector_store = client.vector_stores.create(
    name="multilingual_demo",
    extra_body={
        "provider_id": "qdrant",
        "embedding_model": "ollama/nomic-embed-text:latest"
    }
)

print(f"✅ Created vector store: {vector_store.id}")
print(f"   Embedding model: nomic-embed-text")
print(f"   This model maps similar concepts to similar vectors across languages")

---
## Step 4: Insert Multilingual Documents

**What we're doing:** Uploading all documents (in all languages) to the same vector store.

**Why this matters:** 
- All documents get embedded into the same vector space
- Similar concepts in different languages will have similar vectors
- This enables cross-lingual search

**Expected output:** Confirmation as each document is inserted.

In [ ]:
print("📥 Inserting documents into vector store...\n")

for key, doc in documents.items():
    lang = doc['language']
    topic = doc['topic'].replace('_', ' ').title()
    
    # Add metadata to content for context
    content = f"""Language: {language_names[lang]}
Topic: {topic}

{doc['content']}
"""
    
    # Upload and attach to vector store
    pseudo_file = io.BytesIO(content.encode("utf-8"))
    uploaded_file = client.files.create(
        file=(f"{key}.txt", pseudo_file, "text/plain"),
        purpose="assistants"
    )
    
    client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=uploaded_file.id
    )
    
    print(f"   ✓ {language_names[lang]} - {topic}")

print(f"\n✅ All {len(documents)} documents inserted!")

---
## Step 5: Define Search Helper

**What we're doing:** Creating a helper function that clearly shows which languages the results come from.

**Why:** This makes it easy to verify that cross-lingual search is working.

In [ ]:
def multilingual_search(query, query_language="English", max_results=5):
    """
    Search and display results with language information.
    """
    print(f"\n{'='*70}")
    print(f"🔍 Query ({query_language}): '{query}'")
    print(f"{'='*70}")
    
    results = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query=query,
        max_num_results=max_results,
        extra_body={"search_mode": "vector"}
    )
    
    if not results.data:
        print("   ❌ No results found")
        return
    
    print(f"\n📊 Results (showing language of each match):")
    
    for i, result in enumerate(results.data, 1):
        content = result.content[0].text if result.content else ""
        
        # Extract language from content
        lang_line = content.split('\n')[0] if content else ""
        
        # Get preview of actual content
        lines = content.split('\n')
        preview = ""
        for line in lines[3:]:  # Skip metadata lines
            if line.strip():
                preview = line.strip()[:100]
                break
        
        print(f"\n   {i}. Score: {result.score:.4f}")
        print(f"      {lang_line}")
        print(f"      \"{preview}...\"")

print("✅ Search helper defined!")

---
## Step 6: Cross-Lingual Search - English Query

**What we're doing:** Querying in English and observing which languages appear in results.

**What to expect:**
- Results should include documents in ALL languages (EN, ES, IT)
- Documents about the same topic should appear together
- Scores should be similar for semantically equivalent content

**The magic:** We type in English, but find Spanish and Italian content about the same topic!

In [ ]:
# English query about AI/technology
multilingual_search(
    "artificial intelligence and machine learning in healthcare",
    query_language="English"
)

print("\n💡 Notice: Results include Spanish and Italian documents about AI!")
print("   The embedding model understands the CONCEPT, not just the words.")

In [ ]:
# English query about climate/environment
multilingual_search(
    "renewable energy and climate change solutions",
    query_language="English"
)

print("\n💡 Again, we see results in multiple languages about the same topic!")

---
## Step 7: Cross-Lingual Search - Spanish Query

**What we're doing:** Now querying in Spanish to verify it works both ways.

**What to expect:**
- Spanish query should find English and Italian documents
- Results about the queried topic in ALL languages

In [ ]:
# Spanish query about AI
multilingual_search(
    "inteligencia artificial en la medicina",
    query_language="Spanish"
)

print("\n💡 Spanish query found English content about AI in healthcare!")

In [ ]:
# Spanish query about environment
multilingual_search(
    "energía renovable y cambio climático",
    query_language="Spanish"
)

---
## Step 8: Cross-Lingual Search - Italian Query

**What we're doing:** Testing with Italian queries.

**What to expect:** Italian query finds content in all three languages.

In [ ]:
# Italian query about AI
multilingual_search(
    "intelligenza artificiale nella sanità",
    query_language="Italian"
)

print("\n💡 Italian query successfully finds English and Spanish content!")

---
## Step 9: Why Vector Search Beats Keyword for Multilingual

**What we're doing:** Comparing **keyword search** vs **vector search** across languages to show why vector embeddings are essential for multilingual retrieval.

**Key insight:**
- **Keyword search** only matches literal words -- an English query won't find Spanish or Italian content
- **Vector search** matches by meaning -- an English query finds semantically similar content in any language

In [ ]:
# Use English-only words that have NO cognates in Spanish/Italian
# "healthcare" doesn't appear in Spanish ("salud") or Italian ("sanità")
# "diagnosis" doesn't appear in Spanish ("diagnóstico") or Italian ("diagnosi") as substring
query_en = "healthcare treatment diagnosis"

# Keyword search (English query)
print("=" * 70)
print(f"🔍 KEYWORD search (English): '{query_en}'")
print("=" * 70)

keyword_results = client.vector_stores.search(
    vector_store_id=vector_store.id,
    query=query_en,
    max_num_results=5,
    extra_body={"search_mode": "keyword"}
)

if not keyword_results.data:
    print("   ❌ No results! Keyword search can't match across languages.")
else:
    for i, r in enumerate(keyword_results.data, 1):
        lang_line = r.content[0].text.split('\n')[0] if r.content else ""
        print(f"   {i}. Score: {r.score:.4f} | {lang_line}")

# Vector search (same English query)
print(f"\n{'=' * 70}")
print(f"🔍 VECTOR search (English): '{query_en}'")
print("=" * 70)

vector_results = client.vector_stores.search(
    vector_store_id=vector_store.id,
    query=query_en,
    max_num_results=5,
    extra_body={"search_mode": "vector"}
)

for i, r in enumerate(vector_results.data, 1):
    lang_line = r.content[0].text.split('\n')[0] if r.content else ""
    print(f"   {i}. Score: {r.score:.4f} | {lang_line}")

print("\n💡 Key Insight:")
print("   • Keyword search only finds English documents (literal word matching)")
print("   • Vector search finds ALL languages because it matches by MEANING")
print("   • Words like 'healthcare' don't exist in Spanish/Italian text,")
print("     but the CONCEPT is understood via embeddings")

---
## Step 10: Cross-Language Comparison

**What we're doing:** Running the same conceptual query in all three languages to compare results.

**What to expect:** Similar results regardless of query language, proving true multilingual understanding.

In [ ]:
print("#" * 70)
print("# COMPARISON: Same Concept, Different Query Languages")
print("# Topic: AI in Healthcare")
print("#" * 70)

queries = [
    ("artificial intelligence healthcare", "English"),
    ("inteligencia artificial salud", "Spanish"),
    ("intelligenza artificiale sanità", "Italian")
]

for query, lang in queries:
    multilingual_search(query, query_language=lang, max_results=3)

print("\n" + "=" * 70)
print("💡 Key Insight: All three queries found similar content!")
print("   The embedding model creates a 'universal' semantic space.")
print("=" * 70)

---
## Step 11: Cleanup

**What we're doing:** Removing the test vector store.

In [ ]:
client.vector_stores.delete(vector_store.id)
print(f"🗑️  Deleted vector store: {vector_store.id}")
print(f"\n✅ Multilingual demo complete!")

---
## 📚 Summary

### What We Learned

| Concept | Explanation |
|---------|-------------|
| **Cross-lingual search** | Query in one language, find results in any language |
| **Multilingual embeddings** | Models that map similar concepts to similar vectors regardless of language |
| **Keyword vs Vector** | Keyword search fails across languages; vector search understands meaning |
| **Unified vector space** | All languages share the same semantic space |

### Key Takeaways

1. **One query, many languages**: You don't need to translate your query or maintain separate indices
2. **Vector search is essential**: Keyword search only matches literal words and fails across languages
3. **Semantic understanding**: The model understands meaning, not just words
4. **Easy implementation**: Just use a multilingual embedding model - no extra configuration needed

### Embedding Model Options

| Model | Multilingual Support | Available via Ollama |
|-------|---------------------|---------------------|
| `nomic-embed-text` | Moderate | Yes |
| `nomic-embed-text-v2-moe` | Better | Yes (`ollama pull nomic-embed-text-v2-moe`) |
| `multilingual-e5-large` | Excellent | No (use sentence-transformers) |

### Next Steps

- Try `03_multimodal_demo.ipynb` for image + text search
- Try `04_advanced_features_demo.ipynb` for filtering and thresholds